# TIỀN XỬ LÝ DỮ LIỆU & CHUẨN HÓA (PREPROCESSING & SCALING)

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

# Đọc dữ liệu Ames Housin

In [52]:
train_df = pd.read_csv('data/train.csv')

# Xóa Outliers cực đoan theo kết quả Lab 03 (Id 524 và 1299)

In [53]:
train_df = train_df.drop(train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)].index)

# Tách Biến mục tiêu - Target và Đặc trưng - Features

In [54]:
X = train_df.drop(columns=['Id', 'SalePrice'])
y = np.log1p(train_df['SalePrice']) # Biến đổi Log1p cho SalePrice

# Phân loại biến Số - Numerical và biến Chữ - Categorical

In [55]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Xử lý Missing Values

## Điền missing biến số bằng Median

In [56]:
imputer_num = SimpleImputer(strategy='median')
X[num_cols] = imputer_num.fit_transform(X[num_cols])


## Điền missing biến phân loại bằng 'None'

In [57]:
X[cat_cols] = X[cat_cols].fillna('None')

# Categorical Encoding - Chuyển chữ thành số

In [58]:
X_encoded = pd.get_dummies(X, drop_first=True)

# Chia tập Train / Test (80% / 20%)

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)


# Chuẩn hóa dữ liệu 

In [60]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_encoded.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_encoded.columns)

print(f"HOÀN THÀNH BƯỚC TIỀN XỬ LÝ")
print(f"Kích thước X_train sau mã hóa & chuẩn hóa: {X_train_scaled.shape}")

HOÀN THÀNH BƯỚC TIỀN XỬ LÝ
Kích thước X_train sau mã hóa & chuẩn hóa: (1166, 259)


# FILTER METHODS & WRAPPER METHOD (RFE)

In [61]:
from sklearn.feature_selection import mutual_info_regression, RFE
from sklearn.ensemble import RandomForestRegressor

# FILTER METHOD - CORRELATION
## Tính tương quan của từng thuộc tính với biến mục tiêu y


In [62]:
corr_with_y = X_train_scaled.apply(lambda col: col.corr(y_train)).abs()
selected_corr = corr_with_y[corr_with_y > 0.3].index.tolist()

c:\ProgramData\anaconda3\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\ProgramData\anaconda3\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


# FILTER METHOD - MUTUAL INFORMATION


In [63]:
mi_scores = mutual_info_regression(X_train_scaled, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train_scaled.columns)
selected_mi = mi_series.nlargest(20).index.tolist()

# WRAPPER METHOD - RFE (RECURSIVE FEATURE ELIMINATION)

In [64]:
rfe_selector = RFE(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42),
    n_features_to_select=20
)
rfe_selector.fit(X_train_scaled, y_train)
selected_rfe = X_train_scaled.columns[rfe_selector.support_].tolist()

print(f"Filter (Correlation > 0.3): Chọn {len(selected_corr)} đặc trưng.")
print(f"Filter (Mutual Info Top 20): Chọn {len(selected_mi)} đặc trưng.")
print(f"Wrapper (RFE Top 20): Chọn {len(selected_rfe)} đặc trưng.")


KeyboardInterrupt: 

# EMBEDDED METHODS & HUẤN LUYỆN 3 MÔ HÌNH (SVM, RF, XGB)

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


# EMBEDDED METHOD - LASSO 

In [ ]:
lasso = LassoCV(cv=5, random_state=42).fit(X_train_scaled, y_train)
selected_lasso = X_train_scaled.columns[np.abs(lasso.coef_) > 0.005].tolist()

#  EMBEDDED METHOD - TREE IMPORTANCE 

In [ ]:
rf_importance = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_scaled, y_train)
importances = pd.Series(rf_importance.feature_importances_, index=X_train_scaled.columns)
selected_tree = importances.nlargest(20).index.tolist()


#  ĐỊNH NGHĨA TAP CÁC TẬP ĐẶC TRƯNG CẦN THỬ NGHIỆM 

In [ ]:
feature_sets = {
    'All Features (Baseline - 250+)': X_train_scaled.columns.tolist(),
    'Filter (Correlation > 0.3)': selected_corr,
    'Filter (Mutual Information Top 20)': selected_mi,
    'Wrapper (RFE Top 20)': selected_rfe,
    'Embedded (Lasso L1)': selected_lasso,
    'Embedded (Tree Importance Top 20)': selected_tree
}


# ĐỊNH NGHĨA 3 MÔ HÌNH CƠ BẢN 

In [ ]:
models = {
    'SVR (SVM Regression)': SVR(C=1.0, kernel='rbf'),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

print(f"Embedded (Lasso): Chọn {len(selected_lasso)} đặc trưng.")
print(f"Embedded (Tree Importance): Chọn {len(selected_tree)} đặc trưng.")
print("Đã thiết lập sẵn 3 Mô hình và 6 Phương pháp Feature Selection!")

# ĐÁNH GIÁ CHI TIẾT (R2, RMSE, MAE) & VẼ BIỂU ĐỒ SO SÁNH


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

results = []


# Vòng lặp kiểm thử từng tập đặc trưng với từng mô hình

In [ ]:
for set_name, features in feature_sets.items():
    if len(features) == 0:
        continue
    X_tr = X_train_scaled[features]
    X_te = X_test_scaled[features]
    
    for model_name, model in models.items():
        # Huấn luyện mô hình
        model.fit(X_tr, y_train)
        preds = model.predict(X_te)
        
        # Tính toán metric
        r2 = r2_score(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        
        results.append({
            'Feature Selection Method': set_name,
            'Num Features': len(features),
            'Model': model_name,
            'R2 Score': round(r2, 4),
            'RMSE': round(rmse, 4),
            'MAE': round(mae, 4)
        })



# Xuất bảng kết quả tổng hợp

In [ ]:
results_df = pd.DataFrame(results)
print("=== BẢNG SO SÁNH HIỆU SUẤT CÁC MÔ HÌNH VÀ KỸ THUẬT FEATURE SELECTION ===")
print(results_df.to_string(index=False))



# VẼ BIỂU ĐỒ 1: MỨC ĐỘ QUAN TRỌNG CỦA TOP 10 ĐẶC TRƯNG 

In [ ]:
plt.figure(figsize=(10, 5))
importances.nlargest(10).plot(kind='barh', color='teal').invert_yaxis()
plt.title('HÌNH 1: TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (TREE IMPORTANCE)')
plt.xlabel('Mức độ đóng góp (Importance Score)')
plt.tight_layout()
plt.show()


# VẼ BIỂU ĐỒ 2: SO SÁNH R2 SCORE GIỮA CÁC PHƯƠNG PHÁP CỦA MÔ HÌNH XGBOOST 


In [ ]:
plt.figure(figsize=(12, 5))
xgb_results = results_df[results_df['Model'] == 'XGBoost']
sns.barplot(data=xgb_results, x='R2 Score', y='Feature Selection Method', palette='Blues_r')
plt.title('HÌNH 2: SO SÁNH CHỈ SỐ R2 SCORE CỦA MÔ HÌNH XGBOOST THEO CÁC PHƯƠNG PHÁP FEATURE SELECTION')
plt.xlim(0.7, 0.95)
plt.tight_layout()
plt.show()